# Import

In [1]:
!pip -q install transformers accelerate datasets sentencepiece

# Model

In [2]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("Loaded:", MODEL_NAME)
print("Device:", model.device)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-4B-Instruct-2507
Device: cuda:0


# Util Func

In [3]:
import re

def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def clean_answer(text: str) -> str:
    text = text.strip()
    # 첫 줄만 사용
    text = text.split("\n")[0].strip()
    # 코드블록 같은 이상 출력 잘라내기
    text = text.split("```")[0].strip()
    return text

def generate_text(prompt: str, max_new_tokens: int = 128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # temperature/top_p/top_k 안 씀
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

## QA Func

### With Context

In [4]:
def ask_qa_with_context(context: str, question: str, max_new_tokens: int = 48):
    prompt = f"""다음 문서를 읽고 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[문서]
{context}

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)



### Without Context

In [5]:
def ask_qa_without_context(question: str, max_new_tokens: int = 48):
    prompt = f"""다음 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)

## Predict Search Func

In [6]:
def predict_need_search(question: str, context: str = "", max_new_tokens: int = 4):
    prompt = f"""당신의 임무는 외부 검색 필요 여부만 판단하는 것입니다.

규칙:
1. 최신 정보, 현재 시점 정보, 자주 바뀌는 사실이면 SEARCH
2. 일반 상식, 오래 변하지 않는 사실이면 NO_SEARCH
3. 반드시 SEARCH 또는 NO_SEARCH 중 하나만 출력
4. 설명 금지

[질문]
{question}

[문맥]
{context if context else "없음"}

[판단]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)

    if "[판단]" in text:
        result = text.split("[판단]")[-1].strip()
    else:
        result = text.strip()

    result = result.split()[0].upper()

    if result == "SEARCH":
        return "SEARCH"
    elif result == "NO_SEARCH":
        return "NO_SEARCH"
    return result

# DataSet

In [7]:
from datasets import load_dataset

dataset = load_dataset("KorQuAD/squad_kor_v1", split="validation[:300]")

print(type(dataset))
print(dataset)

README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

<class 'datasets.arrow_dataset.Dataset'>
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})


In [8]:
print("=== Dataset Summary ===")
print(dataset)

print("\n=== Number of Rows ===")
print(len(dataset))

print("\n=== Column Names ===")
print(dataset.column_names)

print("\n=== First Sample ===")
print(dataset[0])

=== Dataset Summary ===
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})

=== Number of Rows ===
300

=== Column Names ===
['id', 'title', 'context', 'question', 'answers']

=== First Sample ===
{'id': '6548850-0-0', 'title': '임종석', 'context': '1989년 2월 15일 여의도 농민 폭력 시위를 주도한 혐의(폭력행위등처벌에관한법률위반)으로 지명수배되었다. 1989년 3월 12일 서울지방검찰청 공안부는 임종석의 사전구속영장을 발부받았다. 같은 해 6월 30일 평양축전에 임수경을 대표로 파견하여 국가보안법위반 혐의가 추가되었다. 경찰은 12월 18일~20일 사이 서울 경희대학교에서 임종석이 성명 발표를 추진하고 있다는 첩보를 입수했고, 12월 18일 오전 7시 40분 경 가스총과 전자봉으로 무장한 특공조 및 대공과 직원 12명 등 22명의 사복 경찰을 승용차 8대에 나누어 경희대학교에 투입했다. 1989년 12월 18일 오전 8시 15분 경 서울청량리경찰서는 호위 학생 5명과 함께 경희대학교 학생회관 건물 계단을 내려오는 임종석을 발견, 검거해 구속을 집행했다. 임종석은 청량리경찰서에서 약 1시간 동안 조사를 받은 뒤 오전 9시 50분 경 서울 장안동의 서울지방경찰청 공안분실로 인계되었다.', 'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?', 'answers': {'text': ['1989년 2월 15일'], 'answer_start': [0]}}


## rough match Func

In [9]:
def rough_match(pred: str, gold: str) -> bool:
    pred_n = normalize_text(pred)
    gold_n = normalize_text(gold)
    return (gold_n in pred_n) or (pred_n in gold_n)

## PipeLine

In [10]:
def qa_with_search_pipeline(question, context):
    decision = predict_need_search(question, context)

    if decision == "SEARCH":
        pred = ask_qa_with_context(context, question)
    else:
        pred = ask_qa_without_context(question)

    return decision, pred

### PipeLine Test

In [11]:
pipeline_correct = 0
pipeline_results = []

for ex in dataset:
    question = ex["question"]
    context = ex["context"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    decision, pred = qa_with_search_pipeline(question, context)

    ok = rough_match(pred, gold)
    pipeline_correct += int(ok)

    pipeline_results.append({
        "question": question,
        "decision": decision,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

pipeline_acc = pipeline_correct / len(dataset)
print(f"Pipeline accuracy: {pipeline_correct}/{len(dataset)} = {pipeline_acc:.3f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Pipeline accuracy: 281/300 = 0.937


### PipeLine Errors

In [12]:
pipeline_errors = [r for r in pipeline_results if not r["correct"]]
print(f"Pipeline wrong cases: {len(pipeline_errors)}")

for i, case in enumerate(pipeline_errors[:10]):
    print(f"\n[Pipeline Error {i+1}]")
    print("Q:", case["question"])
    print("Decision:", case["decision"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Pipeline wrong cases: 19

[Pipeline Error 1]
Q: 국무회의의 심의를 거쳐야 한다는 헌법 제 몇 조의 내용인가?
Decision: SEARCH
Gold: 제89조
Pred: 국무회의의 심의를 거쳐야 한다.

[Pipeline Error 2]
Q: 로널드 레이건 정부 출범 당시 알렉산더 헤이그는 어떤 직책을 맡았는가?
Decision: SEARCH
Gold: 초대 국무장관직
Pred: 알렉산더 헤이그는 로널드 레이건 정부 출범 당시 초대 국무장관을 맡았다.

[Pipeline Error 3]
Q: 미국 군대에서 두번째로 높은 직위는?
Decision: SEARCH
Gold: 미국 육군 부참모 총장
Pred: 미국 육군 부참모총장

[Pipeline Error 4]
Q: 제럴드 포드 대통령 시기 헤이그가 최고사령부의 최고 사령관으로 임명된 곳은 어디인가?
Decision: SEARCH
Gold: 미국 유럽 연합군
Pred: 나토

[Pipeline Error 5]
Q: 1규빗을 미터법으로 환산하면 얼마인가?
Decision: NO_SEARCH
Gold: 45~46cm
Pred: 1규빗은 약 1.83 미터입니다.

[Pipeline Error 6]
Q: 노아의 방주를 상징적 의미로 받아들이는 종교는 무엇인가?
Decision: NO_SEARCH
Gold: 기독교
Pred: 이스라엘 종교, 특히 유대교. 노아의 방주는 유대교에서 인간의 죄와 회개, 그리고 하나님과의 약속을 상징한다.

[Pipeline Error 7]
Q: 현재도 노아의 방주를 역사적 사실로 받아들이는 집단은?
Decision: SEARCH
Gold: 보수적 근본주의계열의 개신교
Pred: 과학과 역사학의 발달 이후에도 상징적 의미로 받아들여지는 기독교의 대부분에서 노아의 방주를 역사적 사실로 받아들이는 것은 아니며, 오히려 과학과 역사학

[Pipeline Error 8]
Q: 유사지질학인 홍수지질학이 근원은?
Decision: SEARCH
Gold: 제칠일안식교


In [13]:
under_search = []
qa_failure = []
over_search = []

for r in pipeline_results:
    if not r["correct"]:
        if r["decision"] == "NO_SEARCH":
            under_search.append(r)
        elif r["decision"] == "SEARCH":
            qa_failure.append(r)

print("UNDER_SEARCH:", len(under_search))
print("QA_FAILURE:", len(qa_failure))

UNDER_SEARCH: 5
QA_FAILURE: 14


## 오류 라벨링

In [14]:
import pandas as pd
import os

# 1. pipeline 오답 추출
pipeline_errors = []
for idx, r in enumerate(pipeline_results):
    if not r["correct"]:
        pipeline_errors.append({
            "dataset_idx": int(idx),
            **r
        })

print("Total pipeline errors:", len(pipeline_errors))

# 2. 분석할 개수 설정
N = min(20, len(pipeline_errors))
analysis_samples = pipeline_errors[:N]

# 3. 최종 라벨링용 테이블 생성 (context 전체 포함)
analysis_table = pd.DataFrame([
    {
        "row_id": i,
        "dataset_idx": int(case["dataset_idx"]),
        "question": case["question"],
        "decision": case["decision"],
        "gold": case["gold"],
        "pred": case["pred"],
        "all_gold_answers": " | ".join(dataset[int(case["dataset_idx"])]["answers"]["text"]),
        "context_full": dataset[int(case["dataset_idx"])]["context"],
        "retrieval_quality": "",   # YES / PARTIAL / NO
        "utilization_type": "",    # IGNORED / PARTIAL_USE / CONTRADICTED / HALLUCINATED / OTHER
        "notes": ""
    }
    for i, case in enumerate(analysis_samples)
])

# 4. 캐글 저장 경로
save_dir = "/kaggle/working"
os.makedirs(save_dir, exist_ok=True)

csv_path = os.path.join(save_dir, "repro_04_failure_analysis_full_context.csv")

# 5. 저장
analysis_table.to_csv(csv_path, index=False, encoding="utf-8-sig")

# 6. 확인
print("\n✅ 저장 완료")
print("저장 경로:", csv_path)
print("\n현재 폴더 파일 목록:")
print(os.listdir(save_dir))

display(analysis_table.head(3))

Total pipeline errors: 19

✅ 저장 완료
저장 경로: /kaggle/working/repro_04_failure_analysis_full_context.csv

현재 폴더 파일 목록:
['.virtual_documents', 'repro_04_failure_analysis_full_context.csv']


,row_id,dataset_idx,question,decision,gold,pred,all_gold_answers,context_full,retrieval_quality,utilization_type,notes
0,0,9,국무회의의 심의를 거쳐야 한다는 헌법 제 몇 조의 내용인가?,SEARCH,제89조,국무회의의 심의를 거쳐야 한다.,제89조,"""내각과 장관들이 소외되고 대통령비서실의 권한이 너무 크다"", ""행보가 비서 본연의...",,,
1,1,12,로널드 레이건 정부 출범 당시 알렉산더 헤이그는 어떤 직책을 맡았는가?,SEARCH,초대 국무장관직,알렉산더 헤이그는 로널드 레이건 정부 출범 당시 초대 국무장관을 맡았다.,초대 국무장관직,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",,,
2,2,15,미국 군대에서 두번째로 높은 직위는?,SEARCH,미국 육군 부참모 총장,미국 육군 부참모총장,미국 육군 부참모 총장,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",,,


### 라밸링 기준
#### retrieval_quality
	•	YES: context 안에 정답 단서가 충분히 있음
	•	PARTIAL: 단서는 있는데 불완전하거나 간접적
	•	NO: context 안에 정답 단서가 거의 없음

#### utilization_type
	•	IGNORED: 답이 있었는데 모델이 안 씀
	•	PARTIAL_USE: 일부만 참고해서 틀림
	•	CONTRADICTED: 문서와 반대로 답함 / 다른 entity 선택
	•	HALLUCINATED: 문서와 무관한 답 생성
	•	OTHER: 애매한 경우

In [15]:
analysis_table.loc[0, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "context explicitly contains '(제89조)'; model outputs quoted clause instead of article number"
]

analysis_table.loc[1, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "context explicitly contains '초대 국무장관직'; model returns a full sentence instead of minimal exact answer"
]

analysis_table.loc[2, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "answer is essentially correct but spacing differs: '부참모총장' vs '부참모 총장'"
]

analysis_table.loc[3, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly says '미국 유럽 연합군 최고사령부'; model answers '나토'"
]

analysis_table.loc[4, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly defines 1 cubit as 45~46cm; model confuses it with 300 cubits ≈ 1.83m"
]

analysis_table.loc[5, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context says most Christianity treats it symbolically; model answers Judaism/Israel religion"
]

analysis_table.loc[6, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "context explicitly says conservative fundamentalist Protestant groups still accept it historically; model gives a vague opposite-style summary"
]

analysis_table.loc[7, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "context explicitly says it originated from Seventh-day Adventism; model outputs generic explanation instead"
]

analysis_table.loc[8, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "question asks total days in ark, but model answers 40 days from rainfall detail"
]

analysis_table.loc[9, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "context explicitly contains '마쓰오카 바키치 함장'; model says it is not mentioned"
]

analysis_table.loc[10, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "answer is essentially correct but spacing differs: '신정부군' vs '신정부 군'"
]

analysis_table.loc[11, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "context explicitly contains '함장 마쓰오카 바키치'; model says not mentioned"
]

analysis_table.loc[12, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly says Aristotle regarded theology as central; model answers physics"
]

analysis_table.loc[13, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly says theology branched from metaphysics; model answers natural philosophy"
]

analysis_table.loc[14, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "answer is correct but returned as a full sentence instead of minimal entity"
]

analysis_table.loc[15, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "model gives English expansion instead of acronym IWPG"
]

analysis_table.loc[16, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "meaning is correct ('9개') but exact gold form is '9가지'"
]

analysis_table.loc[17, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "context says the geological layer consists of various kinds of rocks; model over-selects one specific subset"
]

analysis_table.loc[18, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly states Hyundai founder Chung Ju-yung; model answers unrelated person"
]

analysis_table.head(19)

,row_id,dataset_idx,question,decision,gold,pred,all_gold_answers,context_full,retrieval_quality,utilization_type,notes
0,0,9,국무회의의 심의를 거쳐야 한다는 헌법 제 몇 조의 내용인가?,SEARCH,제89조,국무회의의 심의를 거쳐야 한다.,제89조,"""내각과 장관들이 소외되고 대통령비서실의 권한이 너무 크다"", ""행보가 비서 본연의...",YES,PARTIAL_USE,context explicitly contains '(제89조)'; model ou...
1,1,12,로널드 레이건 정부 출범 당시 알렉산더 헤이그는 어떤 직책을 맡았는가?,SEARCH,초대 국무장관직,알렉산더 헤이그는 로널드 레이건 정부 출범 당시 초대 국무장관을 맡았다.,초대 국무장관직,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",YES,PARTIAL_USE,context explicitly contains '초대 국무장관직'; model ...
2,2,15,미국 군대에서 두번째로 높은 직위는?,SEARCH,미국 육군 부참모 총장,미국 육군 부참모총장,미국 육군 부참모 총장,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",YES,PARTIAL_USE,answer is essentially correct but spacing diff...
3,3,31,제럴드 포드 대통령 시기 헤이그가 최고사령부의 최고 사령관으로 임명된 곳은 어디인가?,SEARCH,미국 유럽 연합군,나토,미국 유럽 연합군,헤이그는 닉슨 대통령이 그를 사성 장군과 육군 부참모로 진급시킬 때 집중 광선과 논...,YES,CONTRADICTED,context explicitly says '미국 유럽 연합군 최고사령부'; mod...
4,4,46,1규빗을 미터법으로 환산하면 얼마인가?,NO_SEARCH,45~46cm,1규빗은 약 1.83 미터입니다.,45~46cm,"노아는 하나님의 명령에 따라 배를 만들고 가족과 정결한 짐승 암수 일곱 마리씩, 부...",YES,CONTRADICTED,context explicitly defines 1 cubit as 45~46cm;...
5,5,49,노아의 방주를 상징적 의미로 받아들이는 종교는 무엇인가?,NO_SEARCH,기독교,"이스라엘 종교, 특히 유대교. 노아의 방주는 유대교에서 인간의 죄와 회개, 그리고 ...",기독교,역사학과 과학이 발달하지 않았던 과거 전통 신학계에서는 근본주의적 시각을 받아들여 ...,YES,CONTRADICTED,context says most Christianity treats it symbo...
6,6,54,현재도 노아의 방주를 역사적 사실로 받아들이는 집단은?,SEARCH,보수적 근본주의계열의 개신교,과학과 역사학의 발달 이후에도 상징적 의미로 받아들여지는 기독교의 대부분에서 노아의...,보수적 근본주의계열의 개신교,역사학과 과학이 발달하지 않았던 과거 전통 신학계에서는 근본주의적 시각을 받아들여 ...,YES,IGNORED,context explicitly says conservative fundament...
7,7,61,유사지질학인 홍수지질학이 근원은?,SEARCH,제칠일안식교,성경에 나오는 노아의 홍수와 관련된 역사적 흔적을 찾기 위한 주장에서 비롯됨.,제칠일안식교,"역사학과 과학의 발달이 더뎠던 고대사회에서는, 성경이 단순한 교리적인 부분 뿐 아니...",YES,IGNORED,context explicitly says it originated from Sev...
8,8,77,노아 가족과 동물들이 노아의 방주에서 버틴 날짜는 총 몇일인가?,SEARCH,375일,40일,375일,"기독교 성경 내용에는 모든 종들을 방주에 태운다고 이야기하고 있으나, 어류나 수중 ...",YES,CONTRADICTED,"question asks total days in ark, but model ans..."
9,9,93,하코다테 전쟁 시 반류마루의 함장의 이름은 무엇인가?,SEARCH,마쓰오카 바키치,문서에서 하코다테 전쟁 시 반류마루의 함장 이름이 언급되지 않았습니다.,마쓰오카 바키치,일련의 하코다테 전쟁은 적아 쌍방의 문서에 마쓰오카 바키치 함장의 능란한 조함 능력...,YES,IGNORED,context explicitly contains '마쓰오카 바키치 함장'; mod...


In [16]:
print("=== Retrieval Quality Counts ===")
print(analysis_table["retrieval_quality"].value_counts())

print("\n=== Utilization Type Counts ===")
print(analysis_table["utilization_type"].value_counts())

=== Retrieval Quality Counts ===
retrieval_quality
YES    19
Name: count, dtype: int64

=== Utilization Type Counts ===
utilization_type
PARTIAL_USE     8
CONTRADICTED    7
IGNORED         4
Name: count, dtype: int64
